In [1]:
import math
from typing import Callable
import numpy as np
import scipy.stats as st

def est_D(probabilidades: list[float]):
    n = len(probabilidades)
    prob = sorted(probabilidades)

    D_p = [((i + 1)/n - prob[i]) for i in range(n)]  # Fe(Y) - F(Y)
    D_m = [(prob[i] - i/n) for i in range(n)]  # F(Y) - Fe(Y)

    d = max(max(D_p), max(D_m))

    return d


def estimar_p_valor_simulaciones_continua(
    d_obs: float,
    gen_prob: Callable[[], list[float]],
    N_SIM: int = 10_000
):
    # Se utiliza el estadístico de Kolmogorov-Smirnov

    p_valor = 0.0
    for _ in range(N_SIM):
        probabilidades = gen_prob()
        d_j = est_D(probabilidades)

        if d_j >= d_obs:
            p_valor += 1

    p_valor = p_valor / N_SIM
    return p_valor

### Ejercicio 10.
Decidir si los siguientes datos corresponden a una distribución Normal:
$$ 91.9, 97.8, 111.4, 122.3, 105.4, 95.0, 103.8, 99.6, 96.6, 119.3, 104.8, 101.7 $$

Calcular una aproximación del *p−valor*.

---

### Análisis
$$
\begin{array}{cl}
    H_0 & = \text{"Los datos provienen de una distribución Normal"} \\
    H_1 & = \text{"Los datos no provienen de una distribución Normal"}
\end{array}
$$

Como se desconoce $\mu$ y $\sigma$. Entonces estimare $\hat{\mu} = \overline{X}(n)$ y $\hat{\sigma} = \sqrt{S^2(n)}$

Asumo $\alpha = 0.05$

In [2]:
muestra = [91.9, 97.8, 111.4, 122.3, 105.4, 95.0, 103.8, 99.6, 96.6, 119.3, 104.8, 101.7]
n = len(muestra)

media = np.mean(muestra)

# Equivalente a la raíz cuadrada de S^2(n)
std_m = np.std(muestra, ddof=1)

prob = [st.norm.cdf(muestra[i], loc=media, scale=std_m) for i in range(n)]

d_obs = est_D(prob)
print("d_obs: ", d_obs)

# Como se estiman 2 parámetros, no se puede utilizar la distribución uniforme,
# es necesario utilizar la dist normal y estimar los parámetros de nuevo
def gen_prob_normales():
    # generar n muestras con la distribución F_H0
    muestra_j = np.random.normal(loc=media, scale=std_m, size=n)
    
    # estimaciones de los parámetros
    media_j = np.mean(muestra_j)
    std_m_j = np.std(muestra_j, ddof=1)
    
    # Calculando las prob utilizando la acumulada
    prob = [st.norm.cdf(muestra_j[i], loc=media_j, scale=std_m_j) for i in range(n)]
    return prob

p_valor = estimar_p_valor_simulaciones_continua(
    d_obs,
    gen_prob_normales
)
print("p_valor: ", p_valor)

d_obs:  0.19638944697995542
p_valor:  0.2214


Como $\text{p-valor} \gt \alpha \equiv 0.2214 \gt 0.05$, entonces no existe evidencia suficiente para rechazar $H_0$. 